# T-Lock Pricing Playground

Interactive pricing, risk analysis, and scenario exploration for Treasury Locks.

**How to use:** Change any parameter in the `TRADE PARAMETERS` cell and re-run from there.

## Sections
1. Setup & trade parameters (change these!)
2. Pricing & Greeks
3. Yield sensitivity surface
4. Repo sensitivity
5. Scenario table (parallel shifts)
6. Portfolio of T-Locks
7. P&L attribution (daily move)
8. Benchmark & specialness
9. JSON round-trip proof

In [ ]:
import sys, os, json

# Add pricebook to path (works from examples/, notebooks/, or repo root)
_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ("examples", "notebooks") else os.getcwd()
sys.path.insert(0, os.path.join(_root, "python"))

import numpy as np
import matplotlib.pyplot as plt
from datetime import date
from dateutil.relativedelta import relativedelta

from pricebook.bond import FixedRateBond
from pricebook.treasury_lock import TreasuryLock, tlock_portfolio_risk, tlock_pnl_attribution
from pricebook.treasury_benchmark import TreasuryBenchmark, SpecialnessProfile, when_issued_bond
from pricebook.discount_curve import DiscountCurve
from pricebook.serialization import to_json, from_json
from pricebook.viz import plot, PlotBuilder

print("pricebook loaded OK")

## 1. Trade Parameters

**Change these and re-run everything below.**

In [ ]:
# ============================================================
# TRADE PARAMETERS — CHANGE THESE
# ============================================================

# Valuation
VAL_DATE     = date(2024, 7, 15)
OIS_RATE     = 0.04          # flat OIS rate

# Bond (10Y Treasury note)
ISSUE_DATE   = date(2024, 2, 15)
MATURITY     = date(2034, 2, 15)
COUPON       = 0.04125       # 4-1/8%

# T-Lock
LOCKED_YIELD = 0.042         # locked yield
EXPIRY       = date(2025, 1, 15)
NOTIONAL     = 10_000_000    # $10M
DIRECTION    = 1             # +1 = payer (lock in yield), -1 = receiver
REPO_RATE    = 0.035         # repo financing rate

# ============================================================
# BUILD (don't change below unless adding features)
# ============================================================
bond = FixedRateBond.treasury_note(ISSUE_DATE, MATURITY, COUPON)
tlock = TreasuryLock(bond, LOCKED_YIELD, EXPIRY, NOTIONAL, DIRECTION, REPO_RATE)
curve = DiscountCurve.flat(VAL_DATE, OIS_RATE)

print(f"Bond:  {COUPON*100:.3f}% {MATURITY} (10Y UST)")
print(f"Lock:  {LOCKED_YIELD*100:.2f}% yield, {'payer' if DIRECTION == 1 else 'receiver'}")
print(f"Size:  ${NOTIONAL:,.0f}")
print(f"Repo:  {REPO_RATE*100:.2f}%")
print(f"Curve: flat {OIS_RATE*100:.1f}%")

## 2. Pricing & Greeks

In [ ]:
result = tlock.price(curve)
greeks = tlock.greeks(curve)
dv01 = tlock.dv01(curve)
repo_sens = tlock.repo_sensitivity(curve)

print(f"{'─'*50}")
print(f"  T-Lock PV:           ${result.value:>14,.2f}")
print(f"  Forward price:        {result.forward_price:>14.6f}")
print(f"  Strike price:         {result.strike_price:>14.6f}")
print(f"  Risk factor:          {result.risk_factor:>14.6f}")
print(f"{'─'*50}")
print(f"  Delta (Pucci Eq 14):  {greeks['delta']:>14.6f}")
print(f"  Gamma (Pucci Eq 16):  {greeks['gamma']:>14.6f}")
print(f"  DV01 (1bp):          ${dv01:>14,.2f}")
print(f"  Repo sens (1bp):     ${repo_sens:>14,.2f}")
print(f"{'─'*50}")

## 3. Default Dashboard + Yield / Repo / Greeks

Using `PlotBuilder` directly on the `TreasuryLock` — no raw matplotlib needed.

In [ ]:
# Full 2x2 dashboard: PV vs yield, delta vs yield, repo sensitivity, summary
plot(tlock, curve)

## 4. Individual Panels

Pick and compose panels via `PlotBuilder`.

In [ ]:
# PV vs yield + repo sensitivity side by side
PlotBuilder(tlock, curve).payoff().comparison().figure()

## 5. Scenario Table

PV and Greeks under parallel yield shifts.

In [ ]:
shifts_bp = [-100, -50, -25, -10, 0, 10, 25, 50, 100]

print(f"{'Shift (bp)':>10s} {'PV ($)':>14s} {'DV01 ($)':>12s} {'Delta':>10s} {'Gamma':>10s}")
print('─' * 60)
for s in shifts_bp:
    c = DiscountCurve.flat(VAL_DATE, OIS_RATE + s / 10_000)
    pv = tlock.price(c).value
    d01 = tlock.dv01(c)
    g = tlock.greeks(c)
    marker = ' ◄── current' if s == 0 else ''
    print(f"{s:>+10d} {pv:>14,.2f} {d01:>12,.2f} {g['delta']:>10.4f} {g['gamma']:>10.4f}{marker}")

## 6. Portfolio of T-Locks

Build a book with different locked yields and directions.

In [ ]:
# ============================================================
# PORTFOLIO — ADD/CHANGE TRADES HERE
# ============================================================
portfolio = [
    TreasuryLock(bond, locked_yield=0.038, expiry=EXPIRY, notional=5_000_000, direction=1, repo_rate=0.035),
    TreasuryLock(bond, locked_yield=0.042, expiry=EXPIRY, notional=10_000_000, direction=1, repo_rate=0.035),
    TreasuryLock(bond, locked_yield=0.045, expiry=EXPIRY, notional=3_000_000, direction=-1, repo_rate=0.030),
]

# Aggregate risk
risk = tlock_portfolio_risk(portfolio, curve)

print("Portfolio Risk Report")
print("=" * 50)
print(f"  Positions:          {risk['n_positions']}")
print(f"  Total PV:          ${risk['total_pv']:>14,.2f}")
print(f"  Total DV01:        ${risk['total_dv01']:>14,.2f}")
print(f"  Total Delta:        {risk['total_delta']:>14,.2f}")
print(f"  Total Gamma:        {risk['total_gamma']:>14,.2f}")
print(f"  Repo Sensitivity:  ${risk['repo_sensitivity']:>14,.2f}")
print(f"  Max Overhedge:     ${risk['max_overhedge']:>14,.2f}")

# Per-trade breakdown
print(f"\n{'Trade':>5s} {'Locked':>8s} {'Dir':>4s} {'Notional':>12s} {'PV':>14s} {'DV01':>10s}")
print('─' * 60)
for i, tl in enumerate(portfolio):
    pv = tl.price(curve).value
    d01 = tl.dv01(curve)
    d = 'Pay' if tl.direction == 1 else 'Rcv'
    print(f"{i+1:>5d} {tl.locked_yield*100:>7.2f}% {d:>4s} ${tl.notional:>11,.0f} ${pv:>13,.2f} ${d01:>9,.2f}")

## 7. P&L Attribution

Decompose the P&L from a yield curve move.

In [ ]:
# ============================================================
# SCENARIO — CHANGE THE SHOCKED CURVE
# ============================================================
SHOCKED_RATE = 0.045   # e.g., +50bp move
NEW_REPO     = 0.040   # repo also moves

curve_t1 = DiscountCurve.flat(VAL_DATE, SHOCKED_RATE)
pnl = tlock_pnl_attribution(tlock, curve, curve_t1, repo_rate_t1=NEW_REPO)

print(f"P&L Attribution (OIS {OIS_RATE*100:.1f}% → {SHOCKED_RATE*100:.1f}%, repo {REPO_RATE*100:.1f}% → {NEW_REPO*100:.1f}%)")
print("=" * 50)
print(f"  Curve P&L:        ${pnl['curve_pnl']:>14,.2f}")
print(f"  Repo P&L:         ${pnl['repo_pnl']:>14,.2f}")
print(f"  Roll (1st order):  {pnl['roll_pnl_first_order']:>14,.6f}")
print(f"  Unexplained:      ${pnl['unexplained']:>14,.2f}")
print(f"  ─────────────────────────────────────")
print(f"  TOTAL:            ${pnl['total']:>14,.2f}")

## 8. Benchmark & Specialness

In [ ]:
# Benchmark tracking
bm = TreasuryBenchmark(
    tenor='10Y', otr_cusip='91282CJL2', otr_bond=bond,
    otr_yield=0.042, ofr_yield=0.0425, specialness_bps=15,
    last_auction=date(2024, 5, 15),
)
print(f"10Y Benchmark")
print(f"  OTR yield:    {bm.otr_yield*100:.3f}%")
print(f"  OFR yield:    {bm.ofr_yield*100:.3f}%")
print(f"  OTR-OFR:      {bm.otr_ofr_spread_bps:.1f}bp")
print(f"  Specialness:  {bm.specialness_bps:.0f}bp ({'SPECIAL' if bm.is_special else 'GC'})")
print(f"  Next auction: {bm.next_auction(VAL_DATE)}")
print(f"  Days to auction: {bm.days_to_next_auction(VAL_DATE)}")

# Specialness decay projection
sp = SpecialnessProfile(
    tenor='10Y', gc_rate=0.045,
    special_rates={1: 0.035, 7: 0.037, 30: 0.040, 90: 0.043},
)
days_to_auc = bm.days_to_next_auction(VAL_DATE) or 60
decay = sp.expected_specialness_decay(days_to_auc, sp.overnight_specialness_bps)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot([d['day'] for d in decay], [d['specialness_bps'] for d in decay], 'o-', lw=2, color='crimson')
ax.axvline(days_to_auc, ls='--', color='gray', label=f'Auction ({days_to_auc}d)')
ax.set_xlabel('Days from now')
ax.set_ylabel('Specialness (bp)')
ax.set_title('Projected Specialness Decay → Auction')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. JSON Round-Trip Proof

JSON → object → price → JSON → object → price. Identical.

In [ ]:
j1 = to_json(tlock)
pv1 = tlock.price(curve).value

tlock_rt = from_json(j1)
j2 = to_json(tlock_rt)
pv2 = tlock_rt.price(curve).value

print(f"PV original:   ${pv1:>14,.6f}")
print(f"PV from JSON:  ${pv2:>14,.6f}")
print(f"PV match:       {abs(pv1 - pv2) < 1e-10}")
print(f"JSON match:     {j1 == j2}")
print()
print("JSON payload:")
print(json.dumps(json.loads(j1), indent=2))